# RAAZICID classifier: train on our own photos (experiment 2)

Same model type and export as `train_classifier.ipynb` (experiment 1, Open Images), but trained and tested
only on **our own photos**, which look like real donations.

**Put the photos in Google Drive** (folder names exactly like this):

```
MyDrive/raazicid/dataset/
  train/   food/ grocery_ration/ clothes/ blankets/ winter_support/ hygiene/
           baby/ household/ school_supplies/ books/ mobility_aids/ emergency_relief/
  test/    (same 12 folders)
```

- `train/`: photos the model learns from. Aim for **100+ per category** (at least 50).
- `test/`: **different items** photographed separately, 20–30 per category. Never the same item as in train,
  otherwise the score looks better than it really is.
- No `test/` folder? Then put the 12 category folders directly in `dataset/` and the notebook keeps 20% aside
  for testing automatically.

JPG, PNG, WEBP and iPhone HEIC all work.

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
The model is saved to `MyDrive/raazicid/model_own/`. Copy `raazicid_classifier.pt` and `labels.json` from there
into `backend/models/`.

In [ ]:
# 1. Settings
ARCH = "mobilenet_v2"      # or "efficientnet_b0" (a bit more accurate, a bit slower). Both work in the backend.
EPOCHS_HEAD, EPOCHS_FULL = 4, 16
IMG_SIZE, SEED = 224, 42
DRIVE_DIR = "/content/drive/MyDrive/raazicid"
DATASET_DIR = f"{DRIVE_DIR}/dataset"
OUT_DIR = f"{DRIVE_DIR}/model_own"
CACHE_DIR = "/content/cache"      # photos are copied here once, smaller, so training is fast

# Category ids must match the categories table (backend/app/seed/data.py)
CATEGORY_IDS = {"food": 1, "grocery_ration": 2, "clothes": 3, "blankets": 4, "winter_support": 5, "hygiene": 6,
                "baby": 7, "household": 8, "school_supplies": 9, "books": 10, "mobility_aids": 11, "emergency_relief": 12}
SLUGS = sorted(CATEGORY_IDS, key=CATEGORY_IDS.get)

!pip -q install pillow-heif
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 2. Read the dataset: count photos, find unreadable files and duplicates between train and test
import glob, hashlib, random
import numpy as np
from PIL import Image, ImageOps
import pillow_heif
pillow_heif.register_heif_opener()

random.seed(SEED); np.random.seed(SEED)
EXT = (".jpg", ".jpeg", ".png", ".webp", ".heic", ".heif")
has_split = os.path.isdir(f"{DATASET_DIR}/train")

def list_photos(root):
    out = {}
    for slug in SLUGS:
        out[slug] = sorted(f for f in glob.glob(f"{root}/{slug}/**/*", recursive=True) if f.lower().endswith(EXT))
    unknown = [d for d in os.listdir(root) if os.path.isdir(f"{root}/{d}") and d not in SLUGS and d not in ("train", "test")] if os.path.isdir(root) else []
    if unknown:
        print("⚠ Folders with unknown names (ignored):", unknown, "→ rename to one of", SLUGS)
    return out

def cache(src, split):
    # Fix phone rotation, shrink to 320 px, save as JPG in /content/cache. Returns None if unreadable.
    h = hashlib.md5(open(src, "rb").read()).hexdigest()
    dst = f"{CACHE_DIR}/{split}/{h}.jpg"
    if not os.path.exists(dst):
        try:
            im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
            im.thumbnail((320, 320)); os.makedirs(os.path.dirname(dst), exist_ok=True); im.save(dst, quality=90)
        except Exception as e:
            print("  ✗ cannot read", src, "-", e); return None, h
    return dst, h

if has_split:
    raw = {"train": list_photos(f"{DATASET_DIR}/train"), "test": list_photos(f"{DATASET_DIR}/test")}
else:
    raw = {"all": list_photos(DATASET_DIR)}

data, hashes = {}, {}
for split, by_slug in raw.items():
    data[split] = {}
    for slug, files in by_slug.items():
        items = []
        for f in files:
            dst, h = cache(f, split)
            if dst:
                items.append(dst); hashes.setdefault(h, set()).add(split)
        data[split][slug] = sorted(set(items))

dupes = [h for h, s in hashes.items() if {"train", "test"} <= s]
if dupes:
    print(f"⚠ {len(dupes)} photos are in BOTH train and test. They are removed from test so the score is honest.")
    for slug in SLUGS:
        data["test"][slug] = [f for f in data["test"][slug] if os.path.basename(f)[:-4] not in dupes]

print(f"\n{'category':17s}" + "".join(f"{s:>8s}" for s in data))
for slug in SLUGS:
    row = "".join(f"{len(data[s][slug]):8d}" for s in data)
    n_train = len(data["train" if has_split else "all"][slug])
    warn = "   ⚠ none yet" if n_train == 0 else "   ⚠ very few" if n_train < 30 else ("   (more photos would help)" if n_train < 80 else "")
    print(f"{slug:17s}{row}{warn}")

In [ ]:
# 3. Split: train / validation (picks the best epoch) / test (final score)
from sklearn.model_selection import train_test_split

def flat(by_slug):
    f, y = [], []
    for i, slug in enumerate(SLUGS):
        f += by_slug[slug]; y += [i] * len(by_slug[slug])
    return np.array(f), np.array(y)

MIN_PHOTOS = 5   # a category needs at least this many photos to be split and learned
src = data["train" if has_split else "all"]
too_few = [s for s in SLUGS if 0 < len(src[s]) < MIN_PHOTOS]
if too_few:
    print(f"⚠ Fewer than {MIN_PHOTOS} photos, left out for now:", too_few)
for s in too_few:
    src[s] = []
present = [i for i, s in enumerate(SLUGS) if len(src[s]) > 0]
missing = [s for s in SLUGS if SLUGS.index(s) not in present]
if missing:
    print("⚠ No training photos for:", missing, "- the model cannot suggest these yet.")

if has_split:
    f_all, y_all = flat(data["train"])
    f_train, f_val, y_train, y_val = train_test_split(f_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED)
    f_test, y_test = flat(data["test"])
    if len(f_test) == 0:
        print("⚠ dataset/test is empty: using 20% of train for testing instead")
        f_train, f_test, y_train, y_test = train_test_split(f_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED)
else:
    f_all, y_all = flat(data["all"])
    f_train, f_tmp, y_train, y_tmp = train_test_split(f_all, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
    f_val, f_test, y_val, y_test = train_test_split(f_tmp, y_tmp, test_size=0.67, stratify=y_tmp, random_state=SEED)
keep = np.isin(y_test, present)   # a category the model never saw cannot be scored fairly
if (~keep).any():
    print(f"⚠ {int((~keep).sum())} test photos left out: their category has no training photos")
f_test, y_test = f_test[keep], y_test[keep]
print(len(f_train), "train /", len(f_val), "validation /", len(f_test), "test photos")

In [ ]:
# 4. Model: pretrained on ImageNet, new last layer for our 12 categories (transfer learning)
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# Strong augmentation: our dataset is small, so show each photo in many variations
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.4, 0.4, 0.4, 0.05),
    transforms.RandomGrayscale(0.05), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class Photos(Dataset):
    def __init__(self, files, labels, tf): self.files, self.labels, self.tf = files, labels, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i): return self.tf(Image.open(self.files[i]).convert("RGB")), int(self.labels[i])

dl = lambda f, y, tf, sh: DataLoader(Photos(f, y, tf), batch_size=32, shuffle=sh, num_workers=2)
train_dl, val_dl = dl(f_train, y_train, train_tf, True), dl(f_val, y_val, eval_tf, False)

if ARCH == "efficientnet_b0":
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(SLUGS))
else:
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
    model.classifier[1] = nn.Linear(model.last_channel, len(SLUGS))
model = model.to(device)

counts = np.bincount(y_train, minlength=len(SLUGS))
weights = torch.tensor(counts.sum() / (len(present) * np.maximum(counts, 1)), dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

In [ ]:
# 5. Train: first only the new layer, then the whole network gently. Keeps the epoch with the best validation accuracy.
import copy, time

def run_epoch(loader, train, opt=None):
    model.train(train); total = correct = 0; loss_sum = 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device); out = model(x); loss = loss_fn(out, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); total += len(y); correct += (out.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total

history, best_acc, best_state = [], -1.0, None
for stage, (epochs, lr, freeze) in enumerate([(EPOCHS_HEAD, 1e-3, True), (EPOCHS_FULL, 2e-4, False)], 1):
    for p in model.features.parameters(): p.requires_grad = not freeze
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        t = time.time(); tr_loss, tr_acc = run_epoch(train_dl, True, opt); va_loss, va_acc = run_epoch(val_dl, False); sched.step()
        history.append({"stage": stage, "train_acc": tr_acc, "val_acc": va_acc, "train_loss": tr_loss, "val_loss": va_loss})
        if va_acc >= best_acc: best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
        print(f"stage {stage} epoch {ep + 1:2d}: train acc {tr_acc:.3f}  val acc {va_acc:.3f}  ({time.time() - t:.0f}s)")
model.load_state_dict(best_state)
print("Best validation accuracy:", round(best_acc, 3))

In [ ]:
# 6. Final test: accuracy, top-3, per-category report, confusion matrix, and the photos it got wrong
import matplotlib.pyplot as plt, pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def predict(files):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(files), 64):
            x = torch.stack([eval_tf(Image.open(f).convert("RGB")) for f in files[i:i + 64]]).to(device)
            out.append(torch.softmax(model(x), 1).cpu())
    return torch.cat(out).numpy()

p = predict(list(f_test)); pred = p.argmax(1)
acc = float((pred == y_test).mean())
top3 = float(np.mean([y_test[i] in np.argsort(-p[i])[:3] for i in range(len(y_test))]))
labels_present = sorted(set(y_test) | set(pred))
names = [SLUGS[i] for i in labels_present]
print(f"== Test ({len(y_test)} photos): accuracy {acc:.1%}, top-3 accuracy {top3:.1%} ==\n")
print(classification_report(y_test, pred, labels=labels_present, target_names=names, zero_division=0))
report = classification_report(y_test, pred, labels=labels_present, target_names=names, zero_division=0, output_dict=True)

cm = confusion_matrix(y_test, pred, labels=range(len(SLUGS)))
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(cm, display_labels=SLUGS).plot(ax=ax, xticks_rotation=60, colorbar=False)
ax.set_title(f"Own photos test (accuracy {acc:.1%})"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_own_test.png", dpi=150); plt.show()

pd.DataFrame(history)[["train_acc", "val_acc"]].plot(title="Accuracy per epoch"); plt.xlabel("epoch")
plt.savefig(f"{OUT_DIR}/training_curve.png", dpi=150); plt.show()

wrong = [i for i in range(len(y_test)) if pred[i] != y_test[i]]
if wrong:
    n = min(16, len(wrong)); cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3.2 * rows))
    for ax, i in zip(np.array(axes).ravel(), wrong[:n]):
        ax.imshow(Image.open(f_test[i])); ax.axis("off")
        ax.set_title(f"is {SLUGS[y_test[i]]}\nAI: {SLUGS[pred[i]]} {p[i][pred[i]]:.0%}", fontsize=9)
    for ax in np.array(axes).ravel()[n:]: ax.axis("off")
    plt.suptitle("Photos the model got wrong"); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/mistakes.png", dpi=120); plt.show()

In [ ]:
# 7. Export for the backend (same format as experiment 1)
import json, datetime
model.eval().cpu()
torch.jit.trace(model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE)).save(f"{OUT_DIR}/raazicid_classifier.pt")
info = {
    "version": datetime.datetime.now(datetime.timezone.utc).strftime(f"{ARCH}-own-%Y%m%d-%H%M"),
    "architecture": f"{ARCH} (ImageNet weights, fine-tuned on own photos)",
    "input_size": IMG_SIZE, "resize": 256, "mean": MEAN, "std": STD,
    "categories": [{"index": i, "slug": s, "category_id": CATEGORY_IDS[s]} for i, s in enumerate(SLUGS)],
    "train_photos": int(len(f_train)), "best_val_accuracy": round(best_acc, 4),
    "metrics": {"own_photos": {"photos": int(len(y_test)), "accuracy": round(acc, 4), "top3_accuracy": round(top3, 4), "per_category": report}},
    "history": history,
    "data": "Own photos (train/test split: " + ("separate folders" if has_split else "automatic 70/10/20") + ")",
}
json.dump(info, open(f"{OUT_DIR}/labels.json", "w"), indent=2, default=float)
print("Saved to", OUT_DIR, ":", os.listdir(OUT_DIR))

In [ ]:
# 8. Try one photo (upload from your computer)
from google.colab import files as colab_files
for name in colab_files.upload():
    pr = predict([name])[0]
    for i in np.argsort(-pr)[:3]: print(f"{SLUGS[i]:17s} {pr[i]:.1%}")